In [ ]:
import ee

# NOTA: "viirs-peru" es el ID del proyecto de Google Cloud vinculado a Earth Engine, el nombre fue ese pero es para todo EARTH ENGINE
# sirve como autenticación para CUALQUIER dataset de Earth Engine (WorldCover, SRTM,
# Sentinel-2, Open Buildings, etc.), no solo para VIIRS.

try:
    ee.Initialize(project="viirs-peru")
except Exception:
    ee.Authenticate()
    ee.Initialize(project="viirs-peru")

print("Google Earth Engine listo")


## ESA World Cover — Mapa de cobertura del suelo

**Fuente:** European Space Agency (Agencia Espacial Europea), producto llamado
"WorldCover". Disponible en Google Earth Engine como `ESA/WorldCover/v100/2020` (versión
del año 2020) y `ESA/WorldCover/v200/2021` (versión del año 2021, más reciente).

**Qué es, en simple:** imagina un mapa de todo el planeta donde cada cuadradito de 10 por
10 metros en el suelo (eso es lo que significa "resolución de 10 metros") ya viene
clasificado con una etiqueta: ¿es bosque? ¿es cultivo? ¿es una zona construida (casas,
calles)? ¿es agua? ¿es suelo pelado? Es como un Google Maps, pero en vez de mostrar calles
y nombres, muestra "qué tipo de superficie hay aquí" para el planeta entero. No hay que
construir nada: es un mapa que la ESA ya hizo y publicó gratis, y nosotros solo leemos el
valor que le corresponde a cada distrito.

**Clases que distingue:** bosque, arbustos, pastizal, cultivo (`cropland`), zona construida
(`built-up`), suelo desnudo o vegetación escasa, nieve/hielo, cuerpos de agua permanentes,
humedal herbáceo, manglar, musgo y liquen. Para este proyecto interesan sobre todo cuatro:
zona construida, cultivo, agua y suelo desnudo.

**Años cubiertos:** 2020 y 2021 son las dos únicas versiones que existen — a diferencia de
SIAF o SIEN, este mapa no se actualiza todos los años, así que un solo valor (el más
reciente disponible, 2021) se repite para todo el rango 2021-2025 del proyecto, igual que
ya se hace con SRTM.

**Por qué importa para anemia:** este mapa da, de forma rápida y sin tener que entrenar
nada, una primera foto de cómo es el territorio de cada distrito: qué tan urbanizado está
(zona construida), cuánta tierra se usa para sembrar (cultivo, complementa al NDVI que ya
tienen), y cuánta agua hay a la vista (complementa a JRC). Sirve como una capa de contexto
territorial más para el Causal Forest — el modelo que intenta explicar por qué el mismo
gasto rinde distinto según el lugar — y también como punto de comparación rápido antes de
construir la segmentación satelital fina (Etapa 2 del proyecto, sobre imágenes Planet
NICFI): si el mapa fino que ustedes construyan no se parece en nada a este de la ESA en las
zonas donde deberían coincidir (por ejemplo, "cuánto es zona construida"), es una señal de
que algo está mal en la segmentación propia.

**Variables a extraer (aún no calculadas):** el plan es el mismo enfoque que ya usaron
con JRC (agua): convertir cada clase en una máscara binaria (por ejemplo, "es zona
construida sí/no" para cada cuadradito) y luego calcular, por distrito, qué fracción del
área total cae en esa clase.

| Variable planeada | Qué mediría |
|---|---|
| `pct_zona_construida` | Fracción del distrito cubierta por casas, calles, construcciones |
| `pct_cultivo` | Fracción del distrito bajo uso agrícola |
| `pct_agua` | Fracción del distrito con cuerpos de agua visibles |
| `pct_suelo_desnudo` | Fracción del distrito sin cobertura vegetal ni construcción |

**Método (planeado):** el mismo que ya usaron para SRTM, JRC y NDVI — cargar el límite
distrital, convertirlo a lotes de 40 distritos (para no exceder el límite de tamaño de
respuesta de Earth Engine), y usar `reduceRegions` para calcular el promedio de cada
máscara binaria dentro de cada polígono distrital.

**¿Usa IA? Sí — pero no la corren ustedes.** A diferencia de SRTM (fórmula fija de
radar) o NDVI (fórmula aritmética `(B8-B4)/(B8+B4)`), este mapa de cobertura sí se genera
con un modelo de machine learning: la ESA entrenó un clasificador (un modelo de tipo
"gradient boosting", una familia de modelos de aprendizaje supervisado) con millones de
puntos de entrenamiento para que aprenda a distinguir esas clases de cobertura a partir de
imágenes de Sentinel-1 y Sentinel-2. Lo importante para el proyecto es que ese modelo ya
está entrenado y su resultado ya está publicado — el equipo no entrena nada acá, solo
reutiliza la salida, igual que se declaró en el documento maestro: la segmentación fina que
sí construirán ustedes mismos (Etapa 2, con U-Net/SegFormer) es la parte nueva; este mapa es
infraestructura ya validada que se reutiliza tal cual.

**Estado actual: no iniciado.** La sección en el notebook solo tiene el encabezado y una
celda vacía — falta escribir el código de extracción (el mismo patrón que SRTM/JRC/NDVI) y
correrlo para los distritos del piloto.


In [ ]:
# ============================================================
# ESA World Cover — extracción de cobertura de suelo por distrito
# ============================================================

# 1. Cargar la imagen (un solo año disponible: 2021, versión v200)
worldcover = ee.ImageCollection("ESA/WorldCover/v200").first()
mapa = worldcover.select("Map")

# 2. Máscaras binarias para las clases de interés
mask_cultivo    = mapa.eq(40).rename("cultivo")
mask_construido = mapa.eq(50).rename("construido")
mask_desnudo    = mapa.eq(60).rename("desnudo")
mask_agua       = mapa.eq(80).rename("agua")

img_worldcover = (mask_cultivo
                   .addBands(mask_construido)
                   .addBands(mask_desnudo)
                   .addBands(mask_agua))

# 3. Función de reducción por lote (mismo patrón que JRC)
def procesar_lote_worldcover(fc_lote):
    return img_worldcover.reduceRegions(
        collection=fc_lote,
        reducer=ee.Reducer.mean(),
        scale=10,       # resolución nativa de World Cover
        tileScale=4
    )

# 4. Procesamiento por lotes de 40 distritos
lista_ubigeos = limite_distrital["UBIGEO"].tolist()
n_distritos = len(lista_ubigeos)
tamano_lote = 40
resultados_wc = []

for i in range(0, n_distritos, tamano_lote):
    lote_ids = lista_ubigeos[i:i + tamano_lote]
    gdf_lote = limite_distrital[limite_distrital["UBIGEO"].isin(lote_ids)]
    fc_lote = geemap.geopandas_to_ee(gdf_lote)
    fc_resultado = procesar_lote_worldcover(fc_lote)

    datos_lote = fc_resultado.reduceColumns(
        ee.Reducer.toList(5),
        ["UBIGEO", "cultivo", "construido", "desnudo", "agua"]
    ).get("list").getInfo()

    resultados_wc.extend(datos_lote)
    print(f"Lote {i//tamano_lote + 1}/{-(-n_distritos//tamano_lote)} — "
          f"distritos {i} a {min(i+tamano_lote, n_distritos)} listo")

# 5. Armar el dataframe final
df_worldcover = pd.DataFrame(
    resultados_wc,
    columns=["ubigeo", "pct_cultivo", "pct_construido", "pct_desnudo", "pct_agua_visible"]
)
print(df_worldcover.shape)
df_worldcover.head()

# 6. Guardar
df_worldcover.to_csv("../data/processed/esa_worldcover_distrital_2021.csv", index=False)